In [ ]:
# === Persiapan lingkungan (hanya modul bawaan Python) ===
# Di Colab: dataset diunduh sekali dari repository bila belum ada.
# Di laptop: folder repo dideteksi otomatis dari working directory.
import csv
import json
import math
import random
import re
from collections import Counter
from pathlib import Path

REPO = "Aldiss1/PBL_SmartReceipt"
BRANCH = "main"
RAW = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/"

FILES = [
    "ML/Dataset_SmartReceipt_Sintetis.csv",
    "ML/Dataset_Katalog_Indonesia_Sintetis_v1.csv",
    "smart_receipt/assets/text_normalization_id.json",
    "ML/uts/hasil_perbandingan.csv",
]


def _find_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "ML" / "Dataset_SmartReceipt_Sintetis.csv").exists():
            return base
    return None


ROOT = _find_root()
if ROOT is None:
    ROOT = Path("/content/pbl")
    missing = [rel for rel in FILES if not (ROOT / rel).exists()]
    if missing:
        import urllib.request

        for rel in missing:
            dest = ROOT / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(RAW + rel, dest)
            print("unduh:", rel)

BASE_CSV = ROOT / "ML" / "Dataset_SmartReceipt_Sintetis.csv"
CATALOG_CSV = ROOT / "ML" / "Dataset_Katalog_Indonesia_Sintetis_v1.csv"
RULES_PATH = ROOT / "smart_receipt" / "assets" / "text_normalization_id.json"
RESULTS_CSV = ROOT / "ML" / "uts" / "hasil_perbandingan.csv"

RULES = json.loads(RULES_PATH.read_text(encoding="utf-8"))
print("Repo            :", ROOT)
print("Kamus singkatan :", len(RULES["abbreviations"]), "entri")
print("Stopword        :", len(RULES["stopwords"]), "entri")
print("Satuan          :", len(RULES["units"]), "entri")


# UTS PBL Machine Learning - SmartReceipt Analyzer
## Model: k-Nearest Neighbors (kNN) (implementasi manual, tanpa library ML)

**Anggota:** Aldi

Notebook ini mengimplementasikan seluruh pipeline **tanpa `scikit-learn`,
`pandas`, maupun `numpy`** - hanya modul bawaan Python (`csv`, `json`, `math`,
`random`, `re`, `collections`, `pathlib`).

| Komponen | Versi library | Versi manual ini |
|---|---|---|
| Pra-pengolahan | `pipeline_utils.TextPreprocessor` | `TextPreprocessor` (salinan identik) |
| TF-IDF `char_wb` (2,4) | `TfidfVectorizer` | `ManualTfidf` |
| Klasifikasi | library | `k-Nearest Neighbors` |
| Metrik | `accuracy_score`, `classification_report` | fungsi `metrics()` |

kNN memilih kategori berdasarkan mayoritas `k=5` tetangga terdekat.
Kemiripan dihitung dengan cosine similarity (dot product dua vektor
TF-IDF yang sudah dinormalisasi L2), dan tetangga yang lebih dekat
diberi bobot lebih besar (`weights='distance'`).

Alur kerja:

```text
Teks struk OCR -> Pra-pengolahan -> TF-IDF char n-gram -> k-Nearest Neighbors (kNN) manual -> Evaluasi
```


### 1. Aturan Pra-Pengolahan

Aturan diambil dari `smart_receipt/assets/text_normalization_id.json`
agar identik dengan aplikasi Flutter. Hanya modul bawaan Python yang dipakai.

In [ ]:
class TextPreprocessor:
    """Pra-pengolahan teks struk, identik dengan versi aplikasi Flutter."""

    currency = re.compile(r"(?:rp\.?\s*)?\d{1,3}(?:[.,]\d{3})+(?:[.,]\d{2})?")
    number_unit = re.compile(
        r"\b\d+\s*(?:x|pcs|pc|pck|ctn|dus|box|botol|bks|bungkus|sachet|sct|"
        r"kg|gr|gram|ml|ltr|liter|ons|papan|rim|set|pack|pak|lusin|mah|gb|tb|"
        r"watt|w|v|volt|s|l|g|ma|mm|cm|m)\b"
    )
    plain_number = re.compile(r"\b\d+\b")
    percent = re.compile(r"\d+\s*%")
    non_alnum = re.compile(r"[^a-z0-9\s]")
    repeated = re.compile(r"(.)\1{2,}")
    spaces = re.compile(r"\s+")

    def __init__(self, rules):
        self.abbreviations = {
            self._key(k): v for k, v in rules["abbreviations"].items()
        }
        self.stopwords = {self._key(v) for v in rules["stopwords"]}
        self.units = {self._key(v) for v in rules["units"]}
        pattern = "|".join(
            re.escape(v) for v in sorted(self.units, key=len, reverse=True)
        )
        self.unit_noise = re.compile(rf"\b(?:x|qty|{pattern})\b")

    def _key(self, value):
        return self.spaces.sub(" ", value.lower()).strip()

    def normalize(self, value):
        text = str(value).lower()
        text = (
            text.replace("\u2019", "'").replace("\u2018", "'")
            .replace("\u201c", '"').replace("\u201d", '"')
            .replace("\u2013", "-").replace("\u2014", "-").replace("\u00d7", "x")
        )
        text = self.percent.sub(" ", text)
        text = self.currency.sub(" ", text)
        text = self.number_unit.sub(" ", text)
        text = self.unit_noise.sub(" ", text)
        text = self.plain_number.sub(" ", text)
        text = self.repeated.sub(lambda m: m.group(1) * 2, text)
        text = self.spaces.sub(" ", self.non_alnum.sub(" ", text)).strip()
        tokens = [t for t in text.split() if len(t) >= 2]
        joined = " ".join(tokens)
        for key, replacement in self.abbreviations.items():
            if " " in key and key in joined:
                joined = joined.replace(key, replacement)
        result = []
        for token in joined.split():
            if len(token) < 2 or token in self.units or token in self.stopwords:
                continue
            result.extend(
                w for w in self.abbreviations.get(token, token).split() if len(w) >= 2
            )
        return " ".join(result)


prep = TextPreprocessor(RULES)

contoh_teks = [
    "INDOMIE GRG SPL 85G",
    "AQUA BTL 600ML Rp 4.500",
    "635110203830 INDOMIE AYAM BAWAN",
    "PARACETAMOL 2PCS",
    "PERTAMAX 5L",
]

for teks in contoh_teks:
    print(f"{teks!r:45} -> {prep.normalize(teks)!r}")


### 2. Muat dan Bagi Data

Pembagian data memakai kolom `split` bawaan dengan *leakage guard*: produk pada
split validasi/uji tidak pernah muncul di data latih.

In [ ]:
def read_rows(path):
    with open(path, encoding="utf-8") as handle:
        return list(csv.DictReader(handle))


def load_splits():
    """Pembagian data identik dengan pipeline_utils.load_splits()."""
    base = read_rows(BASE_CSV)
    catalog = read_rows(CATALOG_CSV)

    def clean(rows):
        out = []
        for row in rows:
            text = prep.normalize(row["receipt_text"])
            if not text:
                continue
            item = dict(row)
            item["text"] = text
            out.append(item)
        return out

    base = clean(base)
    catalog = clean(catalog)

    train = [r for r in base if r["split"] == "train"]
    val = [r for r in base if r["split"] == "val"]
    test = [r for r in base if r["split"] == "test"]

    held_out = {r["nama_produk"] for r in val} | {r["nama_produk"] for r in test}
    train += [r for r in catalog if r["nama_produk"] not in held_out]
    return train, val, test


train, val, test = load_splits()

print(f"Data latih : {len(train):,} baris")
print(f"Data val   : {len(val):,} baris")
print(f"Data uji   : {len(test):,} baris")
print(f"Jumlah kategori target: {len({r['kategori'] for r in train})}")

for baris in train[:3]:
    print(f"  {baris['nama_produk'][:30]:32} -> {baris['text'][:50]!r}")


### 3. Ekstraksi Fitur TF-IDF

Konfigurasi **dikunci sama untuk seluruh anggota**: `char_wb` (2-4 gram),
`sublinear_tf=True`, `max_features=20.000`, normalisasi L2.

Yang ditiru dari `TfidfVectorizer`:

1. Setiap kata diberi spasi pembatas lalu dipecah menjadi n-gram karakter.
2. TF sublinear: `tf = 1 + ln(count)`.
3. IDF halus: `idf = ln((1 + N) / (1 + df)) + 1`.
4. Normalisasi L2 per dokumen.
5. Ambil 20.000 n-gram dengan document frequency tertinggi.


In [ ]:
WHITE = re.compile(r"\s+")


def char_wb_ngrams(text, min_n=2, max_n=4):
    """Tiru perilaku analyzer='char_wb' scikit-learn."""
    text = WHITE.sub(" ", text.lower())
    grams = []
    for word in text.split():
        word = " " + word + " "
        size = len(word)
        for n in range(min_n, max_n + 1):
            offset = 0
            grams.append(word[offset:offset + n])
            while offset + n < size:
                offset += 1
                grams.append(word[offset:offset + n])
            if offset == 0:
                break
    return grams


class ManualTfidf:
    """TF-IDF char n-gram tanpa scikit-learn."""

    def __init__(self, max_features=20000, min_n=2, max_n=4):
        self.max_features = max_features
        self.min_n = min_n
        self.max_n = max_n

    def _ngrams(self, text):
        return char_wb_ngrams(text, self.min_n, self.max_n)

    def fit(self, texts):
        df_counter = Counter()
        for text in texts:
            df_counter.update(set(self._ngrams(text)))
        terms = sorted(df_counter, key=lambda t: (-df_counter[t], t))
        terms = terms[: self.max_features]
        self.vocabulary_ = {term: i for i, term in enumerate(terms)}
        n_docs = len(texts)
        self.idf_ = [
            math.log((1 + n_docs) / (1 + df_counter[t])) + 1.0 for t in terms
        ]
        return self

    def transform(self, texts):
        rows = []
        for text in texts:
            counts = Counter(
                g for g in self._ngrams(text) if g in self.vocabulary_
            )
            vector = {}
            for gram, count in counts.items():
                idx = self.vocabulary_[gram]
                vector[idx] = (1.0 + math.log(count)) * self.idf_[idx]
            norm = math.sqrt(sum(v * v for v in vector.values()))
            if norm > 0:
                vector = {k: v / norm for k, v in vector.items()}
            rows.append(vector)
        return rows


vectorizer = ManualTfidf().fit([r["text"] for r in train])
x_train = vectorizer.transform([r["text"] for r in train])
x_val = vectorizer.transform([r["text"] for r in val])
x_test = vectorizer.transform([r["text"] for r in test])

y_train = [r["kategori"] for r in train]
y_val = [r["kategori"] for r in val]
y_test = [r["kategori"] for r in test]

n_features = len(vectorizer.vocabulary_)
nnz = sum(len(v) for v in x_train)
print(f"Jumlah fitur (n-gram) : {n_features:,}")
print(f"Elemen tak nol latih  : {nnz:,}")
print(f"Kepadatan matriks     : {nnz / (len(x_train) * n_features):.4%}")


### 4. Pembuatan Model - k-Nearest Neighbors

Karena vektor TF-IDF sudah dinormalisasi L2, cosine similarity cukup dihitung
sebagai dot product. Agar tidak membandingkan setiap dokumen uji dengan seluruh
5.569 dokumen latih, dibuat **inverted index** fitur -> daftar dokumen, sehingga
hanya dokumen yang berbagi n-gram yang dihitung skornya.

Parameter: `k=5`, `weights='distance'`, metrik cosine.


In [ ]:
class ManualKNN:
    """k-Nearest Neighbors dengan cosine similarity, tanpa scikit-learn."""

    def __init__(self, k=5, weights="distance"):
        self.k = k
        self.weights = weights

    def fit(self, vectors, labels):
        self.labels_ = labels
        self.index_ = {}
        for i, vector in enumerate(vectors):
            for feature, value in vector.items():
                self.index_.setdefault(feature, []).append((i, value))
        return self

    def predict(self, vectors):
        predictions = []
        for vector in vectors:
            scores = {}
            for feature, value in vector.items():
                for i, train_value in self.index_.get(feature, ()):
                    scores[i] = scores.get(i, 0.0) + value * train_value
            nearest = sorted(scores.items(), key=lambda kv: -kv[1])[: self.k]
            votes = {}
            for i, similarity in nearest:
                label = self.labels_[i]
                weight = similarity if self.weights == "distance" else 1.0
                votes[label] = votes.get(label, 0.0) + weight
            predictions.append(max(votes, key=votes.get))
        return predictions


model = ManualKNN(k=5, weights="distance")
model.fit(x_train, y_train)
print("Model selesai dilatih.")
print(f"Jumlah dokumen latih : {len(x_train):,}")
print(f"Jumlah fitur terindeks: {len(model.index_):,}")


### 5. Evaluasi

Evaluasi pada **data uji** (790 baris, produk tidak pernah dilihat model) dan
**data validasi** (762 baris). Metrik dihitung manual: akurasi,
presisi/recall/F1 macro dan weighted, laporan per kategori, confusion matrix.


In [ ]:
def metrics(y_true, y_pred):
    """Akurasi, presisi, recall, dan F1 (macro + weighted) tanpa sklearn."""
    labels = sorted(set(y_true) | set(y_pred))
    total = len(y_true)
    correct = sum(1 for t, p in zip(y_true, y_pred) if t == p)

    rows = []
    for label in labels:
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == label and p == label)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != label and p == label)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == label and p != label)
        support = sum(1 for t in y_true if t == label)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall
            else 0.0
        )
        rows.append(
            {
                "label": label,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "support": support,
            }
        )

    n_labels = len(rows)
    summary = {
        "Akurasi": correct / total,
        "Presisi (macro)": sum(r["precision"] for r in rows) / n_labels,
        "Recall (macro)": sum(r["recall"] for r in rows) / n_labels,
        "F1-Score (macro)": sum(r["f1"] for r in rows) / n_labels,
        "F1-Score (weighted)": sum(r["f1"] * r["support"] for r in rows) / total,
    }
    return summary, rows


def print_report(summary, rows):
    for key, value in summary.items():
        print(f"{key:20}: {value:.4f}")
    print()
    print(f"{'Kategori':24}{'Presisi':>9}{'Recall':>9}{'F1':>9}{'Support':>9}")
    for r in rows:
        print(
            f"{r['label']:24}{r['precision']:9.3f}{r['recall']:9.3f}"
            f"{r['f1']:9.3f}{r['support']:9d}"
        )


def confusion(y_true, y_pred, labels):
    table = {a: {b: 0 for b in labels} for a in labels}
    for t, p in zip(y_true, y_pred):
        table[t][p] += 1
    return table


def print_confusion(y_true, y_pred, labels):
    table = confusion(y_true, y_pred, labels)
    short = {label: label[:10] for label in labels}
    print("Confusion matrix (baris = aktual, kolom = prediksi)")
    print(" " * 12 + "".join(f"{short[l]:>12}" for l in labels))
    for actual in labels:
        cells = "".join(f"{table[actual][pred]:>12d}" for pred in labels)
        print(f"{short[actual]:12}{cells}")
    benar = sum(table[label][label] for label in labels)
    print(f"\nTebakan benar: {benar} / {len(y_true)}")


In [ ]:
pred_test = model.predict(x_test)
summary_test, rows_test = metrics(y_test, pred_test)
print("=== DATA UJI ===")
print_report(summary_test, rows_test)


In [ ]:
print_confusion(y_test, pred_test, sorted(set(y_test) | set(pred_test)))


In [ ]:
pred_val = model.predict(x_val)
summary_val, rows_val = metrics(y_val, pred_val)
print("=== DATA VALIDASI ===")
print_report(summary_val, rows_val)

print("\nContoh prediksi data uji:")
for row, pred in list(zip(test, pred_test))[:8]:
    status = "OK" if pred == row["kategori"] else "SALAH"
    print(
        f"  [{status:5}] {row['text'][:38]:40}"
        f" aktual={row['kategori'][:18]:20} prediksi={pred}"
    )


### 6. Perbandingan dengan Versi Library

Selisih 0,0000 berarti implementasi manual menghasilkan keputusan yang
sama persis dengan versi `scikit-learn` pada data uji. Selisih kecil
(di bawah 0,01) wajar terjadi karena perbedaan urutan pembulatan dan
cara memecah nilai seri (tie-break) antar implementasi.

In [ ]:
REFERENCE = {
    "Akurasi": 0.7734177215189874,
    "Presisi (macro)": 0.7387051949828102,
    "Recall (macro)": 0.7562618019120413,
    "F1-Score (macro)": 0.7364738842225392,
    "F1-Score (weighted)": 0.7728157230091715,
}

sklearn_metrics = None
if RESULTS_CSV.exists():
    with open(RESULTS_CSV, encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            if row["Model"] == "k-Nearest Neighbors (kNN)":
                sklearn_metrics = {k: float(v) for k, v in row.items() if k != "Model"}
                break

sumber = "hasil_perbandingan.csv"
if sklearn_metrics is None:
    sklearn_metrics = REFERENCE
    sumber = "nilai acuan bawaan notebook"

print(f"Sumber pembanding: {sumber}\n")
print(f"{'Metrik':22}{'scikit-learn':>14}{'manual':>12}{'selisih':>10}")
for key in REFERENCE:
    lib_value = sklearn_metrics[key]
    manual_value = summary_test[key]
    print(f"{key:22}{lib_value:14.4f}{manual_value:12.4f}{manual_value - lib_value:10.4f}")


### 7. Kesimpulan

- kNN manual mencapai akurasi uji tertinggi di antara keempat model, sesuai
  hasil versi library: kemiripan karakter n-gram sangat cocok dengan
  pendekatan berbasis tetangga.
- Biaya utamanya ada di inferensi: seluruh matriks latih harus disimpan dan
  setiap prediksi menghitung kemiripan terhadap ribuan dokumen. Ini kurang
  ideal untuk on-device, berbeda dengan Naive Bayes.
- Nilai `k` mempengaruhi hasil: `k` kecil peka terhadap noise OCR, `k` besar
  cenderung menyeret kategori mayoritas (Makanan).


### 8. Kontribusi Tim

| Anggota | Peran | Kontribusi |
|---|---|---|
| Rakagali | Ketua / Model Naive Bayes | Pipeline bersama, Naive Bayes versi library dan versi manual, laporan |
| Aldi | Model k-Nearest Neighbors | Melatih dan mengevaluasi kNN, analisis nilai k |
| Rijal | Model Support Vector Machine | Melatih dan mengevaluasi SVM, analisis hyperparameter |
| Noval | Model Random Forest | Melatih dan mengevaluasi Random Forest, analisis jumlah pohon |
| Semua | Kolaborasi | Verifikasi pembagian data dan review hasil perbandingan |
